#今日任务，继续昨天未完成任务。
4.随机梯度下降

  即使在我们无法得到解析解的情况下，我们仍然可以有效地训练模型。 在许多任务上，那些难以优化的模型效果要更好。 因此，弄清楚如何训练这些难以优化的模型是非常重要的。
本书中我们用到一种名为梯度下降（gradient descent）的方法， 这种方法几乎可以优化所有深度学习模型。 它通过不断地在损失函数递减的方向上更新参数来降低误差。
  梯度下降最简单的用法是计算损失函数（数据集中所有样本的损失均值） 关于模型参数的导数（在这里也可以称为梯度）。 但实际中的执行可能会非常慢：因为在每一次更新参数之前，我们必须遍历整个数据集。 因此，我们通常会在每次需要计算更新的时候随机抽取一小批样本， 这种变体叫做小批量随机梯度下降（minibatch stochastic gradient descent）。
  在每次迭代中，我们首先随机抽样一个小批量B， 它是由固定数量的训练样本组成的。 然后，我们计算小批量的平均损失关于模型参数的导数（也可以称为梯度）。 最后，我们将梯度乘以一个预先确定的正数𝜂，并从当前参数的值中减掉。
  我们用下面的数学公式来表示这一更新过程（∂表示偏导数）：
$(w,b)\gets(w,b)-\frac{\eta}{|B|}\sum_{i \in B}\partial_{(w,b)}l^{(i)}(w,b)$

  总结一下，算法的步骤如下:（1）初始化模型参数的值，如随机初始化；（2）从数据集中随机抽取小批量样本且在负梯度的方向上更新参数，并不断迭代这一步骤。 对于平方损失和仿射变换，我们可以明确地写成如下形式:
$w\gets w-\frac{\eta}{|B|}\sum_{i \in B}\partial_wl^{(i)}(w,b)=w-\frac{\eta}{|B|}\sum_{i \in B}x^{(i)}(w^Tx^{(i)}+b-y^{(i)})$
$b\gets b-\frac{\eta}{|B|}\sum_{i \in B}\partial_bl^{(i)}(w,b)=b-\frac{\eta}{|B|}\sum_{i \in B}(w^Tx^{(i)}+b-y^{(i)})$

  上式中的𝐰和𝐱都是向量。 在这里，更优雅的向量表示法比系数表示法（如$𝑤^1,𝑤^2,…,𝑤^𝑑$）更具可读性。 |B|表示每个小批量中的样本数，这也称为批量大小（batch size）。 𝜂表示学习率（learning rate）。 批量大小和学习率的值通常是手动预先指定，而不是通过模型训练得到的。 这些可以调整但不在训练过程中更新的参数称为超参数（hyperparameter）。 调参（hyperparameter tuning）是选择超参数的过程。 超参数通常是我们根据训练迭代结果来调整的， 而训练迭代结果是在独立的验证数据集（validation dataset）上评估得到的。
  在训练了预先确定的若干迭代次数后（或者直到满足某些其他停止条件后）， 我们记录下模型参数的估计值，表示为$\dot{w},\dot{b}$。 但是，即使我们的函数确实是线性的且无噪声，这些估计值也不会使损失函数真正地达到最小值。 因为算法会使得损失向最小值缓慢收敛，但却不能在有限的步数内非常精确地达到最小值。
  线性回归恰好是一个在整个域中只有一个最小值的学习问题。 但是对像深度神经网络这样复杂的模型来说，损失平面上通常包含多个最小值。 深度学习实践者很少会去花费大力气寻找这样一组参数，使得在训练集上的损失达到最小。 事实上，更难做到的是找到一组参数，这组参数能够在我们从未见过的数据上实现较低的损失， 这一挑战被称为泛化（generalization）。

5.用模型进行预测

  给定“已学习”的线性回归模型$\dot{w}^Tx+\dot{b}$， 现在我们可以通过房屋面积$x^1$和房龄$x^2$来估计一个（未包含在训练数据中的）新房屋价格。 给定特征估计目标的过程通常称为预测（prediction）或推断（inference）。

  在训练我们的模型时，我们经常希望能够同时处理整个小批量的样本。 为了实现这一点，需要(我们对计算进行矢量化， 从而利用线性代数库，而不是在Python中编写开销高昂的for循环)。

In [1]:
%matplotlib inline
import math
import time
import numpy as np
import torch
from d2l import torch as d2l

  为了说明矢量化为什么如此重要，我们考虑(对向量相加的两种方法)。 我们实例化两个全为1的10000维向量。 在一种方法中，我们将使用Python的for循环遍历向量； 在另一种方法中，我们将依赖对+的调用。

In [2]:
n = 10000
a = torch.ones(n)
b = torch.ones(n)

这里定义一个计时器方便调用

In [3]:
class Timer:  #@save
    """记录多次运行时间"""
    def __init__(self):
        self.times = []
        self.start()

    def start(self):
        """启动计时器"""
        self.tik = time.time()

    def stop(self):
        """停止计时器并将时间记录在列表中"""
        self.times.append(time.time() - self.tik)
        return self.times[-1]

    def avg(self):
        """返回平均时间"""
        return sum(self.times) / len(self.times)

    def sum(self):
        """返回时间总和"""
        return sum(self.times)

    def cumsum(self):
        """返回累计时间"""
        return np.array(self.times).cumsum().tolist()

现在我们可以对工作负载进行基准测试。
首先，我们使用for循环，每次执行一位的加法。

In [4]:
c = torch.zeros(n)
timer = Timer()
for i in range(n):
    c[i] = a[i] + b[i]
f'{timer.stop():.5f} sec'

'0.03797 sec'

第二种方法，使用重载的+运算符来计算按元素的和。

In [5]:
timer.start()
d = a + b
f'{timer.stop():.5f} sec'

'0.00172 sec'